# Early warning for cross-border ping-pong in offshore incident handling
**Replication notebook for "Early Warning of Cross-Border Ping-Pong in Offshore Software Maintenance"**
**Data: BPI Challenge 2013 incidents (Volvo IT), 4TU.ResearchData, doi:10.4121/uuid:500573e6-accc-4b0c-9576-aa5468b10cee**

Maintained by Muhammad Ahmad Zia, The University of Lahore · [ahmadzia.com](https://ahmadzia.com)

This notebook reproduces every table and figure of the paper: label construction with the sender correction,
four definitions of cross-border ping-pong, time-varying Cox models with a proportionality check, eleven prediction
models under incident-grouped repeated cross-validation, common-cohort and ablation analyses, a time-ordered split,
SHAP explanations and single-thread deployment cost. Results are saved to the Drive folder `OSMO_PingPong_replication`.

## How to run (step by step)
1. Open this notebook in Google Colab (File > Upload notebook).
2. Keep the default **CPU** runtime. A GPU is not needed, and nothing here uses one.
3. Click **Runtime > Run all**.
4. When Colab asks to connect to Google Drive, click **Connect to Google Drive**, choose your account and click **Allow**.
5. Wait. The full run takes about 70 to 90 minutes on the free CPU. Each finished piece of work prints a
   time-stamped line, and the same lines are written to `MyDrive/OSMO_PingPong_replication/progress_log.txt`.
6. **Saving is progressive.** Every fold of every model, every temporal-split model and every latency
   measurement is saved to `MyDrive/OSMO_PingPong_replication/` the moment it finishes. Files are written to a
   temporary name first and renamed only when complete, so stopping at any moment never leaves a broken file.
7. **If you stop, or Colab disconnects:** reconnect and click **Runtime > Run all** again. Finished pieces are
   skipped, so you lose at most the one piece that was running.
8. **At the end, a zip downloads automatically** (Step 16). It holds the results summary,
   all result tables, all figures and the progress log. A copy is also kept in `MyDrive/OSMO_PingPong_replication/`.
   If the browser asks whether to allow the download, click **Allow**.
9. **If you want results before the run finishes:** stop the run (**Runtime > Interrupt execution**), scroll to
   the last cell (Step 16), click inside it and press **Shift + Enter**. It zips and downloads whatever is
   finished so far, with a `STATUS.md` file listing what is done and what is still missing.

**What the study asks**
- **RQ1** How common is team ping-pong, how much of it crosses a national border, and how is it associated with resolution time?
- **RQ2** How early can ping-pong and cross-border ping-pong be predicted from the first events of an incident?
- **RQ3** Which factors drive cross-border ping-pong (product, routing team, country, hour of day)?
- **RQ4** What does it cost to deploy each model at a service desk (training time, per-ticket latency, model size)?

## Step 1. Settings
You do not need to change anything here. `FAST_MODE = True` runs a tiny version (1 repeat, 3 folds) if you only want to check the notebook works.

In [ ]:
FAST_MODE = False          # True = quick check only (about 3 minutes)
N_REPEATS = 1 if FAST_MODE else 3
N_FOLDS   = 3 if FAST_MODE else 5
MAX_PREFIX = 10            # prefixes of length 1..10 events
TARGETS = ["pp", "xpp"]    # pp = team ping-pong, xpp = cross-border ping-pong
MODELS = ["Majority", "HistRate", "DecisionTree", "LogReg", "RandomForest", "LightGBM-small",
          "LightGBM-default", "XGBoost-default", "LightGBM", "XGBoost", "CatBoost"]   # LightGBM / XGBoost = tuned per fold on an inner validation split
ALERT_BUDGET = 0.10        # service desk can review the top 10% riskiest incidents
SEED = 42

## Step 2. Connect Google Drive and create the project folders

In [ ]:
import os, sys, pathlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = pathlib.Path('/content/drive/MyDrive/OSMO_PingPong_replication')
except ImportError:                      # running outside Colab (local test)
    BASE = pathlib.Path(os.environ.get('OSMO_BASE', './OSMO_PingPong_replication'))
DATA, RUNS, RES, FIG, MOD = [BASE / d for d in ['data', 'runs', 'results', 'figures', 'models']]
for d in [DATA, RUNS, RES, FIG, MOD]:
    d.mkdir(parents=True, exist_ok=True)
for leftover in BASE.rglob('*.part'):     # half-written files from an earlier interrupted run
    leftover.unlink()

import json, time
LOG = BASE / 'progress_log.txt'
def log(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(line)
    with open(LOG, 'a') as fh:
        fh.write(line + '\n')

def atomic_write(path, writer):
    # write to '<name>.part' first, then rename; a stop mid-write never leaves a half-finished file
    path = pathlib.Path(path); tmp = path.with_name(path.name + '.part')
    writer(tmp); os.replace(tmp, path)

def save_csv(df, path):
    atomic_write(path, lambda p: df.to_csv(p, index=False))

def save_json(obj, path):
    atomic_write(path, lambda p: pathlib.Path(p).write_text(json.dumps(obj, indent=1, default=float)))

log(f'Session started. Everything will be saved under: {BASE}')

## Step 3. Install the extra libraries
Colab already has LightGBM, XGBoost and scikit-learn. This adds CatBoost and SHAP (takes about 1 minute).

In [ ]:
import subprocess, importlib
for pkg in ['catboost', 'shap']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import numpy as np, pandas as pd, time, json, hashlib, platform, joblib, warnings
import lightgbm as lgb, xgboost as xgb, catboost, sklearn, shap, scipy, statsmodels
warnings.filterwarnings('ignore')
env = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
           sklearn=sklearn.__version__, lightgbm=lgb.__version__, xgboost=xgb.__version__,
           catboost=catboost.__version__, shap=shap.__version__, statsmodels=statsmodels.__version__, cpu=platform.processor() or platform.machine(),
           cores=os.cpu_count())
try:
    env['cpu_model'] = [l.split(':')[1].strip() for l in open('/proc/cpuinfo') if 'model name' in l][0]
except Exception:
    pass
json.dump(env, open(RES / 'environment.json', 'w'), indent=2)
print(env)

## Step 4. Download the event log
The log is **BPI Challenge 2013, incidents** (Volvo IT, 7,554 incidents, 65,533 events; Steeman, 2013,
4TU.ResearchData, doi:10.4121/uuid:500573e6-accc-4b0c-9576-aa5468b10cee, CC BY 4.0).
The cell first tries a CSV copy of that log on GitHub and checks its SHA-256 fingerprint. If that fails, it
downloads the original XES file from 4TU. The file is saved to Drive, so this happens only once.

In [ ]:
import urllib.request, gzip, shutil
CSV = DATA / 'BPI2013_incidents.csv'
MIRROR = 'https://media.githubusercontent.com/media/lmu-dbs/BEST/main/data/BPI2013_incidents.csv'
SHA = '76179b9a202f8bfc20d2754bab2152ae7b58ee945dce5d51ac4ad762facd0e94'

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

if not CSV.exists():
    try:
        urllib.request.urlretrieve(MIRROR, CSV)
        assert sha256(CSV) == SHA, 'checksum mismatch'
        print('Downloaded GitHub copy, checksum OK')
    except Exception as e:
        print('GitHub copy failed (', e, ') -> downloading original XES from 4TU')
        if CSV.exists(): CSV.unlink()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pm4py'], check=True)
        import pm4py
        meta = json.load(urllib.request.urlopen(
            'https://data.4tu.nl/v2/articles/0fc5c579-e544-4fab-9143-fab1f5192432/files'))
        url = [m['download_url'] for m in meta if m['name'].endswith(('.xes.gz', '.xes'))][0]
        xes = DATA / 'BPI_Challenge_2013_incidents.xes.gz'
        urllib.request.urlretrieve(url, xes)
        log = pm4py.read_xes(str(xes))
        log.to_csv(CSV, index=False)
else:
    print('Log already in Drive')

## Step 5. Core functions (labels and prefix features)
Nothing to change here. Read the comments if you want to see how ping-pong is defined. Note: the event that opens a new team segment is logged under the person who SENT the incident, so it is no longer used to decide the receiving team's country.

In [ ]:
"""Core pipeline: labels + prefix features for BPIC 2013 incidents (OSMO ping-pong study)."""
import numpy as np
import pandas as pd
from collections import Counter

CASE, TS = "case:concept:name", "ts"
GROUP, RCTY, OCTY = "org:group", "resource country", "organization country"
UNKNOWN_CTY = "0"


def load_log(path):
    df = pd.read_csv(path)
    df[TS] = pd.to_datetime(df["time:timestamp"], utc=True, format="mixed")
    df["org:role"] = df["org:role"].fillna("NA")
    df = df.sort_values([CASE, TS], kind="stable").reset_index(drop=True)
    return df


def _is_sender_event(i, groups, resources):
    """BPIC 2013 logs the event that opens a new team segment (usually 'Awaiting Assignment') under the
    person who transferred the incident, i.e. the same resource as the previous event (92% of team changes).
    Such an event says nothing about where the RECEIVING team is, so it is excluded from that team's country."""
    return i > 0 and groups[i] != groups[i - 1] and resources[i] == resources[i - 1]


def _segments(groups, countries, resources):
    """Collapse consecutive events of the same team into segments.
    Returns list of (team, country_mode, first_event_index); the sender-logged opening event is not
    used for the segment's country."""
    segs = []
    for i, (g, c) in enumerate(zip(groups, countries)):
        if segs and segs[-1][0] == g:
            segs[-1][1].append(c)
        else:
            segs.append([g, [] if _is_sender_event(i, groups, resources) else [c], i])
    out = []
    for g, cs, i0 in segs:
        known = [x for x in cs if x != UNKNOWN_CTY]
        mode = Counter(known).most_common(1)[0][0] if known else None  # ties -> first seen (deterministic)
        out.append((g, mode, i0))
    return out


def case_labels(df):
    """Per-case outcome labels with the event index at which each outcome first occurs.
    pp  : team ping-pong = incident returns to a support team it already left (A..B..A)
    xpp : cross-border ping-pong = a ping-pong return where the returning team's country
          differs from the country of the team it bounced back from
    xh  : any handover between teams located in different countries
    """
    rows = []
    for cid, g in df.groupby(CASE, sort=False):
        segs = _segments(g[GROUP].tolist(), g[RCTY].tolist(), g['org:resource'].tolist())
        pp_pos = xpp_pos = xh_pos = None
        n_bounce = n_xbounce = 0
        pairs = []
        for i in range(1, len(segs)):
            team, cty, ev = segs[i]
            pteam, pcty, _ = segs[i - 1]
            if cty and pcty and cty != pcty and xh_pos is None:
                xh_pos = ev
            if team in [s[0] for s in segs[: i - 1]]:
                n_bounce += 1
                if pp_pos is None:
                    pp_pos = ev
                if cty and pcty and cty != pcty:
                    n_xbounce += 1
                    pairs.append(tuple(sorted((cty, pcty))))
                    if xpp_pos is None:
                        xpp_pos = ev
        rows.append(dict(
            case=cid, n_events=len(g), start=g[TS].iloc[0], end=g[TS].iloc[-1],
            duration_h=(g[TS].iloc[-1] - g[TS].iloc[0]).total_seconds() / 3600,
            n_teams=g[GROUP].nunique(), n_handovers=len(segs) - 1,
            n_countries=len({x for x in g[RCTY] if x != UNKNOWN_CTY}),
            pp=pp_pos is not None, pp_pos=pp_pos, xpp=xpp_pos is not None, xpp_pos=xpp_pos,
            xh=xh_pos is not None, xh_pos=xh_pos, n_bounce=n_bounce, n_xbounce=n_xbounce,
            xpp_pairs=";".join(f"{a}|{b}" for a, b in pairs),
        ))
    return pd.DataFrame(rows).set_index("case")


LIFECYCLES = None  # filled from data
CAT_FEATS = ["impact_last", "product", "org_involved_last", "org_country_first", "first_team",
             "first_country", "last_team", "last_country", "last_role", "last_lifecycle", "last_activity"]


def prefix_features(df, max_k=10):
    """Aggregation encoding of every prefix of length 1..max_k (only info known at event k)."""
    lifecycles = sorted(df["lifecycle:transition"].unique())
    acts = sorted(df["concept:name"].unique())
    recs = []
    for cid, g in df.groupby(CASE, sort=False):
        grp = g[GROUP].to_numpy(); cty = g[RCTY].to_numpy(); ts = g[TS].to_numpy()
        lc = g["lifecycle:transition"].to_numpy(); ac = g["concept:name"].to_numpy()
        res = g["org:resource"].to_numpy(); role = g["org:role"].to_numpy()
        imp = g["impact"].to_numpy(); prod = g["product"].to_numpy()
        orgi = g["organization involved"].to_numpy(); octy = g[OCTY].to_numpy()
        n = len(g)
        t0 = pd.Timestamp(ts[0])
        known_cty = [c if c != UNKNOWN_CTY else None for c in cty]
        for k in range(1, min(n, max_k) + 1):
            # team segments within prefix
            seg_teams, seg_cty = [], []
            for j in range(k):
                if not seg_teams or seg_teams[-1] != grp[j]:
                    seg_teams.append(grp[j])
                    seg_cty.append([] if _is_sender_event(j, grp, res) else [known_cty[j]])
                else:
                    seg_cty[-1].append(known_cty[j])
            seg_mode = []
            for cs in seg_cty:
                kn = [x for x in cs if x]
                seg_mode.append(Counter(kn).most_common(1)[0][0] if kn else None)
            x_hand = sum(1 for i in range(1, len(seg_mode))
                         if seg_mode[i] and seg_mode[i - 1] and seg_mode[i] != seg_mode[i - 1])
            tk = pd.Timestamp(ts[k - 1])
            gaps = np.diff(ts[:k]).astype("timedelta64[s]").astype(float) / 3600 if k > 1 else np.array([0.0])
            last_known = next((c for c in reversed(known_cty[:k]) if c), "UNK")
            first_known = next((c for c in known_cty[:k] if c), "UNK")
            r = dict(
                case=cid, k=k,
                impact_last=imp[k - 1], product=prod[k - 1], org_involved_last=orgi[k - 1],
                org_country_first=octy[0], first_team=grp[0], first_country=first_known,
                last_team=grp[k - 1], last_country=last_known, last_role=role[k - 1],
                last_lifecycle=lc[k - 1], last_activity=ac[k - 1],
                n_teams=len(set(grp[:k])), n_handovers=len(seg_teams) - 1, n_xborder_handovers=x_hand,
                n_countries=len({c for c in known_cty[:k] if c}), n_resources=len(set(res[:k])),
                frac_system_events=float(np.mean(res[:k] == "Siebel")),
                elapsed_h=(tk - t0).total_seconds() / 3600, last_gap_h=float(gaps[-1]),
                mean_gap_h=float(gaps.mean()), max_gap_h=float(gaps.max()),
                start_hour=t0.hour, start_wday=t0.dayofweek, cur_hour=tk.hour, cur_wday=tk.dayofweek,
                impact_changed=int(len(set(imp[:k])) > 1),
            )
            for l in lifecycles:
                r["cnt_lc_" + l.replace(" ", "_")] = int(np.sum(lc[:k] == l))
            for a in acts:
                r["cnt_act_" + a] = int(np.sum(ac[:k] == a))
            recs.append(r)
    X = pd.DataFrame(recs)
    for c in CAT_FEATS:
        X[c] = X[c].astype(str)
    return X


def eligible_prefixes(X, labels, target):
    """Early-warning set: keep prefix k only if the case continues after k (n_events > k)
    and the target outcome has not already happened within the first k events."""
    L = labels[["n_events", target, target + "_pos"]]
    M = X.join(L, on="case")
    pos = M[target + "_pos"]
    keep = (M["n_events"] > M["k"]) & (pos.isna() | (pos >= M["k"]))
    M = M[keep].copy()
    y = M[target].astype(int).to_numpy()
    return M.drop(columns=["n_events", target, target + "_pos"]), y

## Step 5b. Label-validity and resolution-time functions
Four alternative definitions of cross-border ping-pong and a time-varying Cox model.

In [ ]:
"""Label-validity and resolution-time functions: alternative cross-border definitions and time-varying Cox models."""
from collections import Counter
import numpy as np
import pandas as pd

ALT_DEFS = ['D2_team_home', 'D3_dominant75', 'D5_handler']


def sender_mask(df):
    """True for events that open a new team segment but were logged by the person who sent the incident
    (same resource as the previous event). These carry the SENDER's country, not the receiving team's."""
    prev_g = df.groupby(CASE)[GROUP].shift(); prev_r = df.groupby(CASE)['org:resource'].shift()
    return (prev_g.notna() & (df[GROUP] != prev_g) & (df['org:resource'] == prev_r)).to_numpy()


def team_home(df, sender_fix=True):
    """Fixed home country of each team = most frequent known handler country over the whole log."""
    k = df[~sender_mask(df)] if sender_fix else df
    k = k[k[RCTY] != UNKNOWN_CTY]
    home = k.groupby(GROUP)[RCTY].agg(lambda s: Counter(s).most_common(1)[0][0])
    purity = k.groupby(GROUP)[RCTY].agg(lambda s: Counter(s).most_common(1)[0][1] / len(s))
    return home.to_dict(), purity, k.groupby(GROUP).size()


def all_definitions(df, sender_fix=True):
    """Cross-border ping-pong under four definitions, each with the event index where it first occurs.
    D1 segment_mode : most frequent handler country of the returning segment differs from that of the segment
                      it came back from (main label of the paper)
    D2 team_home    : fixed home country of the returning team differs from that of the team it came back from
    D3 dominant75   : D1, counted only if both segments have one country holding >= 75% of known events, no tie
    D5 handler      : country of the last known handler before the hand-back differs from the first known
                      handler after it
    Also returns segment ambiguity counts."""
    home, _, _ = team_home(df, sender_fix)
    snd = sender_mask(df) if sender_fix else np.zeros(len(df), bool)
    rows, seg_stats = {}, Counter()
    for cid, g in df.groupby(CASE, sort=False):
        idx = g.index.to_numpy()
        segs = []
        for i, (grp, c) in enumerate(zip(g[GROUP], g[RCTY])):
            c = None if c == UNKNOWN_CTY else c
            if segs and segs[-1]['team'] == grp:
                segs[-1]['c'].append(c)
            else:
                segs.append(dict(team=grp, c=[] if snd[idx[i]] else [c], ev=i))
        for s in segs:
            kn = [x for x in s['c'] if x]; cnt = Counter(kn).most_common(2)
            s['mode'] = cnt[0][0] if cnt else None
            s['tie'] = len(cnt) == 2 and cnt[0][1] == cnt[1][1]
            s['dom'] = bool(cnt) and cnt[0][1] / len(kn) >= 0.75 and not s['tie']
            s['first'] = kn[0] if kn else None; s['last'] = kn[-1] if kn else None
            seg_stats['segments'] += 1
            seg_stats['no_known_country'] += not kn
            seg_stats['multi_country'] += len(set(kn)) > 1
            seg_stats['tie'] += s['tie']
        r = {k: None for k in ['pp_pos', 'D1_segment_mode_pos'] + [d + '_pos' for d in ALT_DEFS]}
        for i in range(1, len(segs)):
            a, b = segs[i], segs[i - 1]
            if a['team'] not in [s['team'] for s in segs[:i - 1]]:
                continue
            def hit(key, cond):
                if cond and r[key] is None:
                    r[key] = a['ev']
            hit('pp_pos', True)
            hit('D1_segment_mode_pos', a['mode'] and b['mode'] and a['mode'] != b['mode'])
            ha, hb = home.get(a['team']), home.get(b['team'])
            hit('D2_team_home_pos', ha and hb and ha != hb)
            hit('D3_dominant75_pos', a['dom'] and b['dom'] and a['mode'] != b['mode'])
            hit('D5_handler_pos', a['first'] and b['last'] and a['first'] != b['last'])
        for k in list(r):
            r[k[:-4]] = r[k] is not None
        rows[cid] = r
    return pd.DataFrame.from_dict(rows, orient='index'), dict(seg_stats)


def episodes(df, L):
    """Counting-process episodes: each incident is split at the moment of its first ping-pong and its first
    cross-border ping-pong, so exposure is TIME-VARYING (removes the time-at-risk bias of ever/never groups)."""
    t0 = df.groupby(CASE)[TS].first()
    ev_times = df.groupby(CASE)[TS].apply(list)
    rows = []
    for cid, r in L.iterrows():
        T = r.duration_h
        if T <= 0:
            continue
        times = ev_times[cid]
        def onset(pos):
            if pd.isna(pos): return None
            return (times[int(pos)] - t0[cid]).total_seconds() / 3600
        tp, tx = onset(r.pp_pos), onset(r.xpp_pos)
        cuts = sorted({0.0, T, *(c for c in (tp, tx) if c is not None and 0 < c < T)})
        for a, b in zip(cuts[:-1], cuts[1:]):
            rows.append(dict(case=cid, start=a, stop=b, event=int(b == T),
                             pp_any=int(tp is not None and tp <= a), xpp=int(tx is not None and tx <= a)))
    E = pd.DataFrame(rows)
    E['domestic_pp'] = E.pp_any * (1 - E.xpp)
    return E


def cox_time_varying(E, L, covars):
    """Cox model with left-truncated (start, stop] episodes. HR < 1 = slower resolution while exposed."""
    from statsmodels.duration.hazard_regression import PHReg
    D = E.join(L[covars], on='case') if covars else E
    X = pd.get_dummies(D[['domestic_pp', 'xpp'] + covars], columns=covars, drop_first=True, dtype=float)
    X = X.loc[:, X.std() > 0]
    m = PHReg(D.stop.values, X.values, status=D.event.values, entry=D.start.values, ties='efron').fit()
    out = pd.DataFrame({'term': X.columns, 'coef': m.params, 'se': m.bse, 'p': m.pvalues})
    out['HR'] = np.exp(out.coef); out['HR_lo'] = np.exp(out.coef - 1.96 * out.se); out['HR_hi'] = np.exp(out.coef + 1.96 * out.se)
    return out[out.term.isin(['domestic_pp', 'xpp'])]


def cox_time_bands(E, L, covars, cuts=(24.0, 168.0)):
    """Proportionality check: split every episode at the given hours after opening, let the exposure effect differ
    by time band (exposure x band) and test equal cross-border effects across bands with a Wald test."""
    from statsmodels.duration.hazard_regression import PHReg
    bands = [f'0-{int(cuts[0])}h'] + [f'{int(a)}-{int(b)}h' for a, b in zip(cuts[:-1], cuts[1:])] + [f'>{int(cuts[-1])}h']
    rows = []
    for r in E.itertuples(index=False):
        pts = sorted({r.start, r.stop, *(c for c in cuts if r.start < c < r.stop)})
        for a, b in zip(pts[:-1], pts[1:]):
            rows.append(dict(case=r.case, start=a, stop=b, event=int(r.event and b == r.stop), xpp=r.xpp,
                             domestic_pp=r.domestic_pp, band=bands[sum(a >= c for c in cuts)]))
    S = pd.DataFrame(rows)
    D = S.join(L[covars], on='case') if covars else S
    X = pd.DataFrame(index=D.index)
    for b in bands:
        X[f'xpp_{b}'] = D.xpp * (D.band == b); X[f'domestic_pp_{b}'] = D.domestic_pp * (D.band == b)
    if covars:
        X = X.join(pd.get_dummies(D[covars], drop_first=True, dtype=float))
    X = X.loc[:, X.std() > 0].astype(float)
    m = PHReg(D.stop.values, X.values, status=D.event.values, entry=D.start.values, ties='efron').fit()
    out = []
    for j, c in enumerate(X.columns):
        if c.startswith(('xpp_', 'domestic_pp_')):
            exp, band = c.rsplit('_', 1)
            out.append(dict(exposure=exp, band=band, events_while_exposed=int(((X[c] > 0) & (D.event == 1)).sum()),
                            HR=np.exp(m.params[j]), HR_lo=np.exp(m.params[j] - 1.96 * m.bse[j]),
                            HR_hi=np.exp(m.params[j] + 1.96 * m.bse[j]), p=m.pvalues[j]))
    idx = [list(X.columns).index(f'xpp_{b}') for b in bands]
    R = np.zeros((len(bands) - 1, X.shape[1]))
    for i in range(len(bands) - 1):
        R[i, idx[i]], R[i, idx[i + 1]] = 1, -1
    wald_p = float(m.wald_test(R, scalar=True).pvalue)
    return pd.DataFrame(out), wald_p

## Step 6. Build labels and RQ1 descriptive analysis
The resolution-time medians here are NAIVE (ever vs never). Step 6c gives the corrected, time-varying analysis.

In [ ]:
df = load_log(CSV)
print('events', len(df), '| incidents', df[CASE].nunique())
assert df[CASE].nunique() == 7554 and len(df) == 65533, 'Unexpected log size'
LBL = DATA / 'case_labels.pkl'
if LBL.exists():
    L = pd.read_pickle(LBL)
else:
    L = case_labels(df); L.to_pickle(LBL)

from scipy.stats import mannwhitneyu
rows = []
for name, mask in [('All incidents', slice(None)), ('No ping-pong', ~L.pp),
                   ('Domestic ping-pong only', L.pp & ~L.xpp), ('Cross-border ping-pong', L.xpp)]:
    s = L[mask]
    rows.append(dict(group=name, incidents=len(s), share=len(s) / len(L),
                     median_resolution_h=s.duration_h.median(), p75_resolution_h=s.duration_h.quantile(.75),
                     median_events=s.n_events.median(), median_teams=s.n_teams.median()))
T1 = pd.DataFrame(rows)
save_csv(T1, RES / 'T1_prevalence_naive_cost.csv')
print(T1.round(3).to_string(index=False))
u = mannwhitneyu(L.duration_h[L.xpp], L.duration_h[~L.pp], alternative='greater')
print(f'\nNAIVE comparison only (see Step 6c): Mann-Whitney U p = {u.pvalue:.2e}')
print(f'Share of ping-pong incidents that cross a border: {L.xpp.sum() / L.pp.sum():.1%}')
print(f'Incidents with any cross-border team handover: {L.xh.mean():.1%}')

pairs = (L.xpp_pairs[L.xpp_pairs != ''].str.split(';').explode().value_counts()
         .rename_axis('country_pair').reset_index(name='cross_border_bounces'))
save_csv(pairs, RES / 'T2_country_pairs.csv')
print('\nTop country pairs in cross-border ping-pong'); print(pairs.head(12).to_string(index=False))

L['start_hour_utc'] = L.start.dt.hour
hour = L.groupby('start_hour_utc').agg(incidents=('xpp', 'size'), xpp_rate=('xpp', 'mean'), pp_rate=('pp', 'mean')).reset_index()
save_csv(hour, RES / 'T3_rate_by_start_hour.csv')

first_cty = df.groupby(CASE)[RCTY].agg(lambda s: next((c for c in s if c != UNKNOWN_CTY), 'UNK'))
L['first_country'] = first_cty
cty = (L.groupby('first_country').agg(incidents=('xpp', 'size'), xpp_rate=('xpp', 'mean'), pp_rate=('pp', 'mean'))
       .query('incidents >= 100').sort_values('xpp_rate', ascending=False).reset_index())
save_csv(cty, RES / 'T4_rate_by_first_country.csv')
print('\nCross-border ping-pong rate by country of first handling team (>=100 incidents)'); print(cty.round(3).to_string(index=False))
L['start_month'] = L.start.dt.strftime('%Y-%m').where(L.start >= '2012-03-01', 'before 2012-03')
month = L.groupby('start_month').agg(incidents=('xpp', 'size'), xpp_rate=('xpp', 'mean'), pp_rate=('pp', 'mean'),
                                     median_resolution_h=('duration_h', 'median')).reset_index()
save_csv(month, RES / 'T4b_rate_by_opening_month.csv')
print('\nBy opening month (shows the log\'s selection effect: older incidents are mostly long-running)'); print(month.round(3).to_string(index=False))
log('Step 6 descriptive tables saved')

## Step 6b. Label validity
Checks how ambiguous a team segment's country is, with and without the sender fix, and recomputes cross-border
ping-pong under four definitions. If the definitions agree, the label does not depend on one arbitrary choice.

In [ ]:
VAL = RES / 'T0b_label_definitions.csv'
ALTF = DATA / 'alt_labels.pkl'
if VAL.exists() and ALTF.exists():
    T0a = pd.read_csv(RES / 'T0a_segment_ambiguity.csv'); T0b = pd.read_csv(VAL); T0c = pd.read_csv(RES / 'T0c_definition_overlap.csv')
    ALT = pd.read_pickle(ALTF)
else:
    amb_rows, def_rows = [], []
    for fix in [False, True]:
        A, st = all_definitions(df, sender_fix=fix)
        A = A.reindex(L.index)
        _, purity, nev = team_home(df, sender_fix=fix)
        amb_rows.append(dict(sender_fix=fix, segments=st['segments'],
                             pct_multi_country=100 * st['multi_country'] / st['segments'],
                             pct_tie=100 * st['tie'] / st['segments'],
                             pct_no_known_country=100 * st['no_known_country'] / st['segments'],
                             pct_teams_single_country=100 * (purity == 1).mean(),
                             event_weighted_team_purity=float((purity * nev).sum() / nev.sum())))
        for d in ['D1_segment_mode'] + ALT_DEFS:
            def_rows.append(dict(sender_fix=fix, definition=d, incidents=int(A[d].sum()), rate=A[d].mean(),
                                 share_of_pingpong=A[d].sum() / A.pp.sum(),
                                 median_resolution_h=L.duration_h[A[d]].median()))
        if fix:
            assert (A['D1_segment_mode'] == L.xpp).all(), 'D1 must equal the main label'
            ALT = A
            defs = ['D1_segment_mode'] + ALT_DEFS
            T0c = pd.DataFrame([[ (A[a] & A[b]).sum() / (A[a] | A[b]).sum() for b in defs] for a in defs],
                               index=defs, columns=defs).reset_index().rename(columns={'index': 'definition'})
    T0a = pd.DataFrame(amb_rows); T0b = pd.DataFrame(def_rows)
    save_csv(T0a, RES / 'T0a_segment_ambiguity.csv'); save_csv(T0c, RES / 'T0c_definition_overlap.csv')
    atomic_write(ALTF, lambda q: ALT.to_pickle(q))
    save_csv(T0b, VAL)
    log('Step 6b label validity saved')
print(T0a.round(3).to_string(index=False)); print(); print(T0b.round(4).to_string(index=False)); print(); print(T0c.round(2).to_string(index=False))

## Step 6c. Does cross-border ping-pong slow resolution? (time-varying Cox model)
Comparing incidents that EVER bounced with those that never did is biased: incidents open longer have more chances
to bounce, and in this log older incidents are mostly long-running. Here each incident is split at the moment it first
bounces, and resolution speed is compared before vs after, with and without adjustment for impact, organisation line,
first country and opening period. Hazard ratio (HR) below 1 = slower resolution after the bounce.

In [ ]:
COX = RES / 'T1c_cox_time_varying.csv'
first = df.groupby(CASE).first()
def top(s, n=100): return s.where(s.map(s.value_counts()) >= n, 'Other')
L['impact_final'] = df.groupby(CASE)['impact'].last()
L['org_line'] = top(first['organization involved'])
L['first_cty'] = top(first[RCTY])
L['start_week'] = L.start.dt.tz_convert(None).dt.to_period('W').astype(str).where(L.start >= '2012-04-01', 'before Apr 2012')
if COX.exists():
    T1c = pd.read_csv(COX)
else:
    E = episodes(df, L)
    specs = [('unadjusted', None, []),
             ('+ impact', None, ['impact_final']),
             ('+ impact, org line, first country', None, ['impact_final', 'org_line', 'first_cty']),
             ('+ opening month', None, ['start_month']),
             ('+ impact, org line, first country, opening week', None, ['impact_final', 'org_line', 'first_cty', 'start_week']),
             ('opened from 2012-04-01 only, + impact, org line, first country', '2012-04-01', ['impact_final', 'org_line', 'first_cty']),
             ('opened from 2012-05-01 only, + impact, org line, first country', '2012-05-01', ['impact_final', 'org_line', 'first_cty'])]
    rows = []
    for name, since, cov in specs:
        Ls = L if since is None else L[L.start >= since]
        Es = E[E.case.isin(Ls.index)]
        o = cox_time_varying(Es, Ls, cov)
        for _, r in o.iterrows():
            rows.append(dict(specification=name, exposure=r.term, incidents=len(Ls), HR=r.HR, HR_lo=r.HR_lo, HR_hi=r.HR_hi, p=r.p))
    T1c = pd.DataFrame(rows); save_csv(T1c, COX); log('Step 6c Cox model saved')
print(T1c.round(4).to_string(index=False))

## Step 6d. Proportional-hazards check by time band
Every episode is split at 24 and 168 hours after opening, the effect of a bounce may differ in each band, and a Wald
test checks whether the cross-border effect is the same in all three bands.

In [ ]:
PHB = RES / 'T1d_cox_time_bands.csv'
if PHB.exists():
    T1d = pd.read_csv(PHB)
else:
    if 'E' not in globals():
        E = episodes(df, L)
    rows = []
    for name, since, cov in [('+ opening month', None, ['start_month']),
                             ('+ impact, org line, first country, opening week', None, ['impact_final', 'org_line', 'first_cty', 'start_week']),
                             ('opened from 2012-04-01 only, + impact, org line, first country', '2012-04-01', ['impact_final', 'org_line', 'first_cty'])]:
        Ls = L if since is None else L[L.start >= since]
        bands, wald_p = cox_time_bands(E[E.case.isin(Ls.index)], Ls, cov)
        bands.insert(0, 'specification', name); bands['wald_p_equal_xpp_effect'] = wald_p
        rows.append(bands)
    T1d = pd.concat(rows, ignore_index=True); save_csv(T1d, PHB); log('Step 6d time-band check saved')
print(T1d.round(4).to_string(index=False))

## Step 7. Prefix features (what the service desk knows after k events)

In [ ]:
XF = DATA / 'prefix_features.pkl'
if XF.exists():
    X = pd.read_pickle(XF)
else:
    t = time.time(); X = prefix_features(df, MAX_PREFIX); X.to_pickle(XF)
    print(f'built in {time.time() - t:.0f}s')
print('prefix rows', X.shape)
# fix category lists once (category names only, no labels) so every fold sees the same codes
CAT_LEVELS = {c: sorted(X[c].unique()) for c in CAT_FEATS}

## Step 8. Model definitions
Eleven models: majority and rule baselines, a single decision tree, logistic regression, random forest, a small LightGBM, default and tuned LightGBM/XGBoost, and CatBoost. Every model sees only what is known at prefix k.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

def as_cat(M):
    M = M.copy()
    for c in CAT_FEATS:
        if c in M.columns:
            M[c] = pd.Categorical(M[c], categories=CAT_LEVELS[c])
    return M

class Majority:
    def fit(self, X, y): self.p = float(np.mean(y)); return self
    def predict_proba(self, X): return np.column_stack([1 - np.full(len(X), self.p), np.full(len(X), self.p)])

class HistRate:
    # rule baseline: smoothed historical outcome rate of the first team and of the product (training data only)
    def __init__(self, m=20): self.m = m
    def fit(self, X, y):
        self.prior = float(np.mean(y)); self.maps = {}
        for c in ['first_team', 'product']:
            g = pd.DataFrame({'c': X[c].astype(str).values, 'y': y}).groupby('c').y.agg(['sum', 'count'])
            self.maps[c] = ((g['sum'] + self.m * self.prior) / (g['count'] + self.m)).to_dict()
        return self
    def predict_proba(self, X):
        p = np.mean([X[c].astype(str).map(self.maps[c]).fillna(self.prior).values for c in self.maps], axis=0)
        return np.column_stack([1 - p, p])

LGB_GRID = {
    'default': dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_data_per_group=50, cat_smooth=10),
    'reg_light': dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=50, min_data_per_group=100,
                      cat_smooth=50, cat_l2=10, max_cat_threshold=16),
    'reg_strong': dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=200, min_data_per_group=300,
                       cat_smooth=200, cat_l2=50, max_cat_threshold=8, reg_lambda=10, colsample_bytree=0.6),
}
XGB_GRID = {
    'default': dict(n_estimators=400, learning_rate=0.05, max_depth=6),
    'reg_light': dict(n_estimators=400, learning_rate=0.05, max_depth=4, min_child_weight=20, reg_lambda=10, max_cat_threshold=16),
    'reg_strong': dict(n_estimators=500, learning_rate=0.03, max_depth=5, min_child_weight=50, reg_lambda=20, max_cat_threshold=8, gamma=1),
}

def make_lgb(cfg, seed):
    p = dict(subsample=0.8, subsample_freq=1, colsample_bytree=0.8); p.update(LGB_GRID[cfg])
    return lgb.LGBMClassifier(**p, random_state=seed, verbose=-1)

def make_xgb(cfg, seed):
    p = dict(subsample=0.8, colsample_bytree=0.8); p.update(XGB_GRID[cfg])
    return xgb.XGBClassifier(**p, tree_method='hist', enable_categorical=True, max_cat_to_onehot=1,
                             random_state=seed, n_jobs=-1, eval_metric='logloss')

class InnerTuned:
    # nested tuning: pick the config with the best AUC on an inner, case-grouped 75/25 split of the
    # training data only, then refit that config on all training data
    def __init__(self, builder, grid, seed, groups=None):
        self.builder, self.grid, self.seed, self.groups = builder, grid, seed, groups
    def fit(self, X, y):
        from sklearn.model_selection import StratifiedGroupKFold
        g = self.groups if self.groups is not None else np.arange(len(y))
        itr, iva = next(StratifiedGroupKFold(4, shuffle=True, random_state=self.seed).split(X, y, g))
        self.inner_auc = {}
        for cfg in self.grid:
            m = self.builder(cfg, self.seed).fit(X.iloc[itr], y[itr])
            self.inner_auc[cfg] = roc_auc_score(y[iva], m.predict_proba(X.iloc[iva])[:, 1])
        self.best = max(self.inner_auc, key=self.inner_auc.get)
        self.est = self.builder(self.best, self.seed).fit(X, y)
        return self
    def predict_proba(self, X): return self.est.predict_proba(X)

def make_model(name, num_cols, seed, groups=None):
    if name == 'LightGBM-default': return make_lgb('default', seed)
    if name == 'LightGBM-small':     # deployable small model: 60 trees of 7 leaves, strong categorical regularisation
        p = dict(LGB_GRID['reg_strong']); p.update(n_estimators=60, num_leaves=7)
        return lgb.LGBMClassifier(**p, subsample=0.8, subsample_freq=1, random_state=seed, verbose=-1)
    if name == 'DecisionTree':       # single interpretable tree, depth 6
        from sklearn.tree import DecisionTreeClassifier
        pre = ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FEATS),
            ('num', 'passthrough', num_cols)])
        return make_pipeline(pre, DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=seed))
    if name == 'XGBoost-default': return make_xgb('default', seed)
    if name == 'LightGBM': return InnerTuned(make_lgb, LGB_GRID, seed, groups)
    if name == 'XGBoost': return InnerTuned(make_xgb, XGB_GRID, seed, groups)
    if name == 'Majority': return Majority()
    if name == 'HistRate': return HistRate()
    if name == 'LogReg':
        pre = ColumnTransformer([
            ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20), CAT_FEATS),
            ('num', make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), num_cols)])
        return make_pipeline(pre, LogisticRegression(max_iter=3000, C=0.5))
    if name == 'RandomForest':
        pre = ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FEATS),
            ('num', 'passthrough', num_cols)])
        return make_pipeline(pre, RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=seed))
    if name == 'CatBoost':
        return CatBoostClassifier(iterations=500, learning_rate=0.08, depth=6, cat_features=CAT_FEATS,
                                  random_seed=seed, verbose=0, thread_count=-1)
    raise ValueError(name)

def prep(name, M):
    # each model gets the input format it expects
    if name.startswith(('LightGBM', 'XGBoost')): return as_cat(M)
    return M

def unwrap(model):
    return model.est if isinstance(model, InnerTuned) else model

def single_thread(model):
    # used only for the latency measurement, so every model is timed on one CPU thread
    m = unwrap(model)
    last = m.steps[-1][1] if hasattr(m, 'steps') else m
    if hasattr(last, 'n_jobs') and not isinstance(last, CatBoostClassifier): last.set_params(n_jobs=1)
    return m   # CatBoost gets thread_count=1 at prediction time instead

## Step 9. RQ2: repeated, case-grouped cross-validation (resumable)
All prefixes of one incident stay in the same fold, so the model is never tested on an incident it has seen. Each finished fold is saved to Drive.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
DATASETS = {}
for target in TARGETS:
    M, y = eligible_prefixes(X, L, target)
    M = M.reset_index(drop=True)
    DATASETS[target] = (M, y)
    print(f'{target}: {len(y)} prefixes from {M.case.nunique()} incidents, positive rate {y.mean():.3f}')
FEATS = [c for c in DATASETS[TARGETS[0]][0].columns if c != 'case']
NUM_COLS = [c for c in FEATS if c not in CAT_FEATS]

def save_npz(path, **arrays):
    def w(p):
        with open(p, 'wb') as fh:
            np.savez(fh, **arrays)
    atomic_write(path, w)

def done(path):
    # a saved fold counts as done only if it opens cleanly; a damaged file is deleted and recomputed
    if not path.exists():
        return False
    try:
        with np.load(path) as z:
            z['p']
        return True
    except Exception:
        path.unlink(); return False

t_all = time.time(); total = len(TARGETS) * N_REPEATS * len(MODELS) * N_FOLDS
for target in TARGETS:
    M, y = DATASETS[target]
    for rep in range(N_REPEATS):
        cv = StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED + rep)
        folds = list(cv.split(M, y, M['case']))
        for name in MODELS:
            for f, (tr, te) in enumerate(folds):
                out = RUNS / f'{target}__{name}__r{rep}__f{f}.npz'
                if done(out):
                    continue
                Xtr, Xte = prep(name, M.iloc[tr][FEATS]), prep(name, M.iloc[te][FEATS])
                model = make_model(name, NUM_COLS, SEED + rep, groups=M['case'].iloc[tr].values)
                t = time.time(); model.fit(Xtr, y[tr]); fit_s = time.time() - t
                p = model.predict_proba(Xte)[:, 1]
                p_tr = model.predict_proba(Xtr)[:, 1]
                k_tr = M['k'].to_numpy()[tr]
                train_auc = roc_auc_score(y[tr], p_tr) if p_tr.std() > 0 else 0.5
                thr_k1 = float(np.quantile(p_tr[k_tr == 1], 1 - ALERT_BUDGET))   # alert threshold set on TRAINING scores only
                if rep == 0 and f == 0:
                    atomic_write(MOD / f'{target}__{name}.joblib', lambda q: joblib.dump(model, q))
                save_npz(out, idx=te, p=p, fit_s=fit_s, best=getattr(model, 'best', ''), train_auc=train_auc, thr_k1=thr_k1)
                n_done = len(list(RUNS.glob('*__r*__f*.npz')))
                log(f'CV {target} rep{rep} fold{f} {name:16s} fit {fit_s:6.1f}s  saved  [{n_done}/{total} pieces done]')
log('Cross-validation complete')

## Step 10. Collect metrics, per-prefix results and significance tests

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import scipy.stats
from scipy.stats import friedmanchisquare, wilcoxon

def budget_metrics(y, p, budget=ALERT_BUDGET):
    n = max(1, int(round(budget * len(p))))
    top = np.argsort(-p, kind='stable')[:n]
    tp = y[top].sum()
    return tp / n, tp / max(1, y.sum())          # precision@budget, recall@budget

fold_rows, k_rows, pooled_rows, OOF = [], [], [], {}
for target in TARGETS:
    M, y = DATASETS[target]; k = M['k'].to_numpy()
    for name in MODELS:
        for rep in range(N_REPEATS):
            oof = np.full(len(y), np.nan); fit_total = 0
            for f in range(N_FOLDS):
                z = np.load(RUNS / f'{target}__{name}__r{rep}__f{f}.npz')
                idx, p = z['idx'], z['p']; oof[idx] = p; fit_total += float(z['fit_s'])
                yy = y[idx]
                prec, rec = budget_metrics(yy, p)
                kte = k[idx] == 1; flag = p[kte] >= float(z['thr_k1']); y1 = yy[kte]
                fold_rows.append(dict(target=target, model=name, rep=rep, fold=f,
                                      auc=roc_auc_score(yy, p), pr_auc=average_precision_score(yy, p),
                                      brier=brier_score_loss(yy, p), prec_at_budget=prec, rec_at_budget=rec,
                                      fit_s=float(z['fit_s']), train_auc=float(z['train_auc']),
                                      k1_n=int(len(y1)), k1_pos=int(y1.sum()), k1_flagged=int(flag.sum()),
                                      k1_tp=int((flag & (y1 == 1)).sum())))
            for kk in range(1, MAX_PREFIX + 1):
                s = k == kk
                if s.sum() > 50 and 5 < y[s].sum() < s.sum() - 5:
                    prec, rec = budget_metrics(y[s], oof[s])
                    k_rows.append(dict(target=target, model=name, rep=rep, k=kk, n=int(s.sum()), pos_rate=y[s].mean(),
                                       auc=roc_auc_score(y[s], oof[s]), pr_auc=average_precision_score(y[s], oof[s]),
                                       prec_at_budget=prec, rec_at_budget=rec))
            pooled_rows.append(dict(target=target, model=name, rep=rep, fit_total_s=fit_total))
            OOF[(target, name, rep)] = oof
FOLD = pd.DataFrame(fold_rows); KRES = pd.DataFrame(k_rows)
save_csv(FOLD, RES / 'fold_metrics.csv'); save_csv(KRES, RES / 'per_prefix_metrics.csv')

T5 = (FOLD.groupby(['target', 'model'])
      .agg(auc_mean=('auc', 'mean'), auc_sd=('auc', 'std'), pr_auc_mean=('pr_auc', 'mean'), pr_auc_sd=('pr_auc', 'std'),
           brier=('brier', 'mean'), prec_at_10pct=('prec_at_budget', 'mean'), rec_at_10pct=('rec_at_budget', 'mean'),
           fit_s_per_fold=('fit_s', 'mean')).reset_index())
T5['model'] = pd.Categorical(T5['model'], MODELS); T5 = T5.sort_values(['target', 'model'])
save_csv(T5, RES / 'T5_cv_results.csv')
print(T5.round(3).to_string(index=False))

T6 = (KRES.groupby(['target', 'model', 'k']).agg(n=('n', 'first'), pos_rate=('pos_rate', 'first'),
       auc=('auc', 'mean'), pr_auc=('pr_auc', 'mean'), rec_at_10pct=('rec_at_budget', 'mean')).reset_index())
T6['positives'] = (T6.n * T6.pos_rate).round().astype(int)

# overfitting evidence: training vs held-out AUC
T13 = FOLD.groupby(['target', 'model']).agg(train_auc=('train_auc', 'mean'), test_auc=('auc', 'mean')).reset_index()
T13['gap'] = T13.train_auc - T13.test_auc
T13['model'] = pd.Categorical(T13['model'], MODELS); T13 = T13.sort_values(['target', 'model'])
save_csv(T13, RES / 'T13_overfitting_gap.csv'); print(); print(T13.round(3).to_string(index=False))

# operating point at the first event, threshold = top 10% of TRAINING scores (no test outcomes used)
T11 = FOLD.groupby(['target', 'model'])[['k1_n', 'k1_pos', 'k1_flagged', 'k1_tp']].sum().reset_index()
T11 = T11.assign(flagged_pct=T11.k1_flagged / T11.k1_n, precision=T11.k1_tp / T11.k1_flagged.clip(lower=1),
                 recall=T11.k1_tp / T11.k1_pos)
for c in ['k1_n', 'k1_pos', 'k1_flagged', 'k1_tp']: T11[c] = T11[c] / N_REPEATS      # per repeat (each incident once)
T11['model'] = pd.Categorical(T11['model'], MODELS); T11 = T11.sort_values(['target', 'model'])
save_csv(T11, RES / 'T11_operating_point_k1.csv'); print(); print(T11.round(3).to_string(index=False))

# common cohort: the SAME incidents evaluated at k = 1, 3 and 5
cc_rows = []
for target in TARGETS:
    M, y = DATASETS[target]; kk = M['k'].to_numpy()
    elig = M.groupby('case')['k'].apply(set)
    cohort = set(elig.index[elig.apply(lambda s: {1, 3, 5} <= s)])
    inC = M['case'].isin(cohort).to_numpy()
    for name in MODELS:
        if name == 'Majority': continue
        for kq in [1, 3, 5]:
            sel = inC & (kk == kq)
            aucs = [roc_auc_score(y[sel], OOF[(target, name, r)][sel]) for r in range(N_REPEATS)]
            cc_rows.append(dict(target=target, model=name, k=kq, incidents=int(sel.sum()), pos_rate=y[sel].mean(), auc=np.mean(aucs)))
T12 = pd.DataFrame(cc_rows); save_csv(T12, RES / 'T12_common_cohort.csv')
print(); print(T12.pivot_table(index=['target', 'model'], columns='k', values='auc').round(3).to_string())
save_csv(T6, RES / 'T6_per_prefix.csv')

# significance: Friedman over models on fold AUCs, then Wilcoxon best model vs each other (Holm-corrected)
sig_rows = []
for target in TARGETS:
    piv = FOLD[FOLD.target == target].pivot_table(index=['rep', 'fold'], columns='model', values='auc')
    learners = [m for m in MODELS if m != 'Majority']
    fr = friedmanchisquare(*[piv[m] for m in learners])
    best = piv[learners].mean().idxmax()
    pv = []
    for m in learners:
        if m == best: continue
        pv.append((m, wilcoxon(piv[best], piv[m]).pvalue))
    pv.sort(key=lambda t: t[1]); n = len(pv); running = 0
    for i, (m, p) in enumerate(pv):
        running = max(running, min(1, p * (n - i)))
        d = (piv[best] - piv[m]).to_numpy(); J = len(d)
        t_nb = d.mean() / np.sqrt((1 / J + 1 / (N_FOLDS - 1)) * d.var(ddof=1)) if d.var(ddof=1) > 0 else np.inf
        nb_p = 2 * scipy.stats.t.sf(abs(t_nb), J - 1)
        sig_rows.append(dict(target=target, friedman_p=fr.pvalue, best=best, vs=m, wilcoxon_p=p, holm_p=running,
                             nadeau_bengio_p=nb_p, auc_diff=piv[best].mean() - piv[m].mean()))
T7 = pd.DataFrame(sig_rows); save_csv(T7, RES / 'T7_significance.csv')
tune_rows = []
for target in TARGETS:
    for name in ['LightGBM', 'XGBoost']:
        for rep in range(N_REPEATS):
            for f in range(N_FOLDS):
                tune_rows.append(dict(target=target, model=name, config=str(np.load(RUNS / f'{target}__{name}__r{rep}__f{f}.npz')['best'])))
TUNE = pd.DataFrame(tune_rows).value_counts().rename('folds').reset_index().sort_values(['target', 'model', 'folds'], ascending=[True, True, False])
save_csv(TUNE, RES / 'T7b_tuning_choices.csv')
print(); print(TUNE.to_string(index=False))
print(); print(T7.round(4).to_string(index=False))
log('Step 10 metrics saved')

## Step 11. Robustness: time-ordered split
Train on incidents that started first (80%), test on the most recent 20%. The log over-represents long-running old incidents, so the positive rate shifts between train and test; this checks that the models still work under that shift.

In [ ]:
# each (target, model) result is saved as its own small file the moment it finishes
for target in TARGETS:
    M, y = DATASETS[target]
    st = M['case'].map(L.start); cut = st.quantile(0.8)
    tr = (st < cut).to_numpy(); te = ~tr
    for name in MODELS:
            piece = RUNS / f'temporal__{target}__{name}.json'
            if piece.exists():
                continue
            model = make_model(name, NUM_COLS, SEED, groups=M['case'][tr].values)
            model.fit(prep(name, M[tr][FEATS]), y[tr]); p = model.predict_proba(prep(name, M[te][FEATS]))[:, 1]
            p_tr = model.predict_proba(prep(name, M[tr][FEATS]))[:, 1]
            prec, rec = budget_metrics(y[te], p)
            k1 = (M['k'].to_numpy()[te] == 1); k1_tr = (M['k'].to_numpy()[tr] == 1)
            thr = float(np.quantile(p_tr[k1_tr], 1 - ALERT_BUDGET))      # threshold from TRAINING scores only
            flag = p[k1] >= thr; yk = y[te][k1]
            bins = np.clip((p * 10).astype(int), 0, 9)                   # expected calibration error, 10 bins
            ece = sum(abs(y[te][bins == b].mean() - p[bins == b].mean()) * (bins == b).mean() for b in range(10) if (bins == b).any())
            save_json(dict(target=target, model=name, train_pos_rate=y[tr].mean(), test_pos_rate=y[te].mean(),
                           auc=roc_auc_score(y[te], p), pr_auc=average_precision_score(y[te], p),
                           auc_k1=roc_auc_score(y[te][k1], p[k1]), prec_at_10pct=prec, rec_at_10pct=rec,
                           brier=brier_score_loss(y[te], p), ece=float(ece), mean_pred=float(p.mean()),
                           k1_flagged_pct=float(flag.mean()), k1_precision=float(yk[flag].mean()) if flag.any() else 0.0,
                           k1_recall=float(flag[yk == 1].mean())), piece)
            log(f'Temporal split {target} {name} saved')
T8 = pd.DataFrame([json.loads((RUNS / f'temporal__{t}__{n}.json').read_text()) for t in TARGETS for n in MODELS])
save_csv(T8, RES / 'T8_temporal_split.csv')
print(T8.round(3).to_string(index=False))

## Step 11b. Robustness checks (resumable)
Tuned LightGBM, 5 case-grouped folds each:
- **Alternative labels:** cross-border ping-pong defined by team home country (D2), dominant country (D3) and the
  actual handler-to-handler handover (D5).
- **Opening window:** only incidents opened from 1 April 2012 (the dense part of the log).
- **Feature-group ablation:** initial routing only, prefix dynamics only, and both combined.

In [ ]:
INITIAL = ['product', 'first_team', 'first_country', 'org_country_first', 'start_hour', 'start_wday', 'impact_last', 'org_involved_last']
DYNAMIC = [c for c in FEATS if c not in INITIAL]
Lalt = L[['n_events']].join(ALT[[c for d in ALT_DEFS for c in (d, d + '_pos')]])
ANALYSES = {f'def_{d}': dict(labels=Lalt, target=d, feats=FEATS, since=None) for d in ALT_DEFS}
ANALYSES['window_from_2012-04-01'] = dict(labels=L, target='xpp', feats=FEATS, since='2012-04-01')
ANALYSES['ablation_initial_routing_only'] = dict(labels=L, target='xpp', feats=INITIAL, since=None)
ANALYSES['ablation_prefix_dynamics_only'] = dict(labels=L, target='xpp', feats=DYNAMIC, since=None)
ANALYSES['ablation_combined'] = dict(labels=L, target='xpp', feats=FEATS, since=None)
rob_rows = []
for aname, a in ANALYSES.items():
    M, y = eligible_prefixes(X, a['labels'], a['target']); M = M.reset_index(drop=True)
    if a['since']:
        keep = M['case'].map(L.start >= a['since']).to_numpy(); M = M[keep].reset_index(drop=True); y = y[keep]
    num = [c for c in a['feats'] if c not in CAT_FEATS]
    oof = np.full(len(y), np.nan)
    for f, (tr, te) in enumerate(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(M, y, M['case'])):
        out = RUNS / f'robust__{aname}__f{f}.npz'
        if not done(out):
            model = make_model('LightGBM', num, SEED, groups=M['case'].iloc[tr].values)
            model.fit(as_cat(M.iloc[tr][a['feats']]), y[tr])
            save_npz(out, idx=te, p=model.predict_proba(as_cat(M.iloc[te][a['feats']]))[:, 1])
            log(f'Robustness {aname} fold{f} saved')
        with np.load(out) as z:
            oof[z['idx']] = z['p']
    k = M['k'].to_numpy()
    rob_rows.append(dict(analysis=aname, target=a['target'], incidents=M['case'].nunique(), prefixes=len(y),
                         pos_rate=y.mean(), auc=roc_auc_score(y, oof), pr_auc=average_precision_score(y, oof),
                         auc_k1=roc_auc_score(y[k == 1], oof[k == 1]), auc_k5=roc_auc_score(y[k == 5], oof[k == 5])))
T14 = pd.DataFrame(rob_rows); save_csv(T14, RES / 'T14_robustness.csv')
print(T14.round(3).to_string(index=False))

## Step 12. RQ3: what drives cross-border ping-pong (SHAP on LightGBM)

In [ ]:
SH = RES / 'T9_shap_importance_xpp.csv'
if SH.exists():
    T9 = pd.read_csv(SH)
else:
    M, y = DATASETS['xpp']
    m = unwrap(make_model('LightGBM', NUM_COLS, SEED, groups=M['case'].values).fit(as_cat(M[FEATS]), y))
    samp = as_cat(M[FEATS]).sample(min(5000, len(M)), random_state=SEED)
    sv = shap.TreeExplainer(m).shap_values(samp)
    sv = sv[1] if isinstance(sv, list) else sv
    T9 = pd.DataFrame({'feature': FEATS, 'mean_abs_shap': np.abs(sv).mean(0)}).sort_values('mean_abs_shap', ascending=False)
    # which first-team / product values push risk up the most
    for c in ['first_team', 'product', 'first_country']:
        j = FEATS.index(c)
        d = pd.DataFrame({c: samp[c].astype(str).values, 'shap': sv[:, j]}).groupby(c).shap.agg(['mean', 'size'])
        save_csv(d[d['size'] >= 30].sort_values('mean', ascending=False).reset_index(), RES / f'T9_shap_by_{c}.csv')
    save_csv(T9, SH)          # written last: its presence means the whole SHAP step finished
    log('SHAP step saved')
print(T9.head(15).round(4).to_string(index=False))

## Step 13. RQ4: deployment cost at the service desk
Per-ticket (batch-of-one) latency, batch throughput and model size for each model, all on one CPU thread so the comparison is fair.

In [ ]:
# each model's measurement is saved as its own small file the moment it finishes
M, y = DATASETS['xpp']
one_raw = M[FEATS].iloc[[0]]; batch_raw = M[FEATS].sample(min(10000, len(M)), random_state=SEED, replace=len(M) < 10000)
for name in MODELS:
        piece = RUNS / f'deploy__{name}.json'
        if piece.exists():
            continue
        mfile = MOD / f'xpp__{name}.joblib'
        if not mfile.exists():   # safety net: refit if the saved model is missing
            mm = make_model(name, NUM_COLS, SEED, groups=M['case'].values).fit(prep(name, M[FEATS]), y)
            atomic_write(mfile, lambda q: joblib.dump(mm, q))
        model = single_thread(joblib.load(mfile))
        size_file = MOD / f'_size_{name}.joblib'; atomic_write(size_file, lambda q: joblib.dump(model, q))
        pf = (lambda d: model.predict_proba(d, thread_count=1)) if isinstance(model, CatBoostClassifier) else model.predict_proba
        one, batch = prep(name, one_raw), prep(name, batch_raw)
        for _ in range(20): pf(one)
        lat = []
        for _ in range(300):
            t = time.perf_counter(); pf(one); lat.append((time.perf_counter() - t) * 1000)
        t = time.perf_counter(); pf(batch); bt = time.perf_counter() - t
        save_json(dict(model=name, single_row_ms_median=np.median(lat), single_row_ms_p95=np.percentile(lat, 95),
                       batch_rows_per_s=len(batch) / bt, model_size_kb=os.path.getsize(size_file) / 1024,
                       fit_s_per_fold=FOLD[(FOLD.target == 'xpp') & (FOLD.model == name)].fit_s.mean()), piece)
        log(f'Deployment cost {name} saved')
T10 = pd.DataFrame([json.loads((RUNS / f'deploy__{n}.json').read_text()) for n in MODELS])
# one-row calls per second vs rows per second in a 10,000-row batch: the gap is per-call overhead (batch-of-one penalty)
T10['batch_of_one_calls_per_s'] = 1000 / T10.single_row_ms_median
T10['batching_speedup'] = T10.batch_rows_per_s / T10.batch_of_one_calls_per_s
save_csv(T10, RES / 'T10_deployment_cost.csv')
print(T10.round(3).to_string(index=False))

## Step 14. Figures for the paper (saved as 300-dpi PNG in Drive)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
                     'axes.edgecolor': '#888888', 'savefig.dpi': 300, 'savefig.bbox': 'tight'})
COL = {'HistRate': '#2a78d6', 'LogReg': '#eb6834', 'RandomForest': '#1baf7a', 'LightGBM': '#eda100',
       'XGBoost': '#e87ba4', 'CatBoost': '#6250d6'}
MARK = {'HistRate': 'o', 'LogReg': 's', 'RandomForest': '^', 'LightGBM': 'D', 'XGBoost': 'v', 'CatBoost': 'P'}
TNAME = {'pp': 'Team ping-pong', 'xpp': 'Cross-border ping-pong'}
INK = '#333333'

def save_fig(fig, name):
    atomic_write(FIG / name, lambda p: fig.savefig(p, format='png'))
    log(f'Figure saved: {name}')

# Fig 1: country pairs
top = pairs.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.barh(top.country_pair.str.title().str.replace('|', ' – '), top.cross_border_bounces, color='#2a78d6', height=0.65)
ax.set_xlabel('Cross-border ping-pong bounces'); ax.grid(axis='y', visible=False)
ax.set_title('Country pairs involved in cross-border ping-pong', loc='left', color=INK)
save_fig(fig, 'fig1_country_pairs.png'); plt.show()

# Fig 2: hazard ratio of resolution after cross-border ping-pong, by model specification
d = T1c[T1c.exposure == 'xpp'].iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.errorbar(d.HR, range(len(d)), xerr=[d.HR - d.HR_lo, d.HR_hi - d.HR], fmt='o', color='#2a78d6', ms=5, capsize=3, lw=1.5)
ax.axvline(1, color='#999999', lw=1, ls='--')
ax.set_yticks(range(len(d))); ax.set_yticklabels(d.specification, fontsize=7.5); ax.grid(axis='y', visible=False)
ax.set_xlabel('Hazard ratio of resolution after cross-border ping-pong (95% CI)\n< 1 = slower resolution')
ax.set_title('Resolution speed after cross-border ping-pong', loc='left', color=INK)
save_fig(fig, 'fig2_cox_hazard_ratio.png'); plt.show()

# Fig 3: cross-border ping-pong rate by start hour (UTC)
fig, ax = plt.subplots(figsize=(5, 2.8))
h = hour[hour.incidents >= 30]
ax.plot(h.start_hour_utc, h.xpp_rate * 100, color='#2a78d6', lw=2, marker='o', ms=4)
ax.set_xlabel('Hour the incident was opened (UTC)'); ax.set_ylabel('Cross-border ping-pong (%)')
ax.set_xticks(range(0, 24, 3)); ax.set_title('Cross-border ping-pong by opening hour', loc='left', color=INK)
save_fig(fig, 'fig3_rate_by_hour.png'); plt.show()

# Fig 4: AUC by prefix length (small multiples, one per target)
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, target in zip(axes, TARGETS):
    for name in ['HistRate', 'LogReg', 'RandomForest', 'LightGBM', 'XGBoost', 'CatBoost']:
        d = T6[(T6.target == target) & (T6.model == name)]
        ax.plot(d.k, d.auc, color=COL[name], marker=MARK[name], lw=2, ms=5, label=name)
    ax.set_title(TNAME[target], loc='left', color=INK); ax.set_xlabel('Events observed so far (prefix length k)')
    ax.set_xticks(range(1, MAX_PREFIX + 1))
axes[0].set_ylabel('ROC-AUC')
axes[1].legend(frameon=False, fontsize=8, loc='lower left', ncol=2)
save_fig(fig, 'fig4_auc_by_prefix.png'); plt.show()

# Fig 5: recall at alert budget, first event only (k = 1)
M, y = DATASETS['xpp']; s = M['k'].to_numpy() == 1
budgets = np.linspace(0.02, 0.5, 25)
fig, ax = plt.subplots(figsize=(5, 3))
for name in ['HistRate', 'LogReg', 'LightGBM', 'CatBoost']:
    oof = np.full(len(y), np.nan)
    for f in range(N_FOLDS):
        z = np.load(RUNS / f'xpp__{name}__r0__f{f}.npz'); oof[z['idx']] = z['p']
    rec = [budget_metrics(y[s], oof[s], b)[1] for b in budgets]
    ax.plot(budgets * 100, np.array(rec) * 100, color=COL[name], marker=MARK[name], markevery=4, lw=2, ms=5, label=name)
ax.plot(budgets * 100, budgets * 100, color='#999999', lw=1, ls='--', label='Random')
ax.set_xlabel('Incidents flagged at first event (%)'); ax.set_ylabel('Cross-border ping-pong caught (%)')
ax.set_title('Early-warning recall vs review budget', loc='left', color=INK); ax.legend(frameon=False, fontsize=8)
save_fig(fig, 'fig5_alert_budget.png'); plt.show()

# Fig 6: SHAP importance
NICE = {'first_team': 'First assigned team', 'product': 'Product', 'org_country_first': 'Requesting org. country',
        'first_country': 'First handler country', 'last_team': 'Current team', 'start_hour': 'Opening hour',
        'impact_last': 'Impact', 'n_countries': 'Countries so far', 'last_role': 'Current role',
        'last_lifecycle': 'Current status', 'n_teams': 'Teams so far', 'last_country': 'Current handler country',
        'org_involved_last': 'Organisation line', 'start_wday': 'Opening weekday', 'n_handovers': 'Handovers so far',
        'n_xborder_handovers': 'Cross-border handovers so far', 'elapsed_h': 'Hours since opening',
        'max_gap_h': 'Longest gap between events (h)', 'mean_gap_h': 'Mean gap between events (h)',
        'n_resources': 'Handlers so far', 'cur_hour': 'Current hour', 'cur_wday': 'Current weekday', 'k': 'Events so far'}
t9 = T9.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.barh(t9.feature.map(lambda f: NICE.get(f, f)), t9.mean_abs_shap, color='#2a78d6', height=0.65); ax.grid(axis='y', visible=False)
ax.set_xlabel('Mean |SHAP| (log-odds)'); ax.set_title('Drivers of cross-border ping-pong', loc='left', color=INK)
save_fig(fig, 'fig6_shap.png'); plt.show()

# Fig 7: per-ticket latency (log scale)
d = T10[~T10.model.isin(['Majority', 'LightGBM-default', 'XGBoost-default'])].sort_values('single_row_ms_median')
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.barh(d.model, d.single_row_ms_median, color='#2a78d6', height=0.6); ax.set_xscale('log'); ax.grid(axis='y', visible=False)
for i, v in enumerate(d.single_row_ms_median): ax.text(v, i, f' {v:.2f} ms', va='center', color=INK, fontsize=8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
ax.set_xlabel('Median latency per incident, batch of one, single thread (ms, log scale)')
ax.set_title('Per-ticket scoring cost', loc='left', color=INK)
save_fig(fig, 'fig7_latency.png'); plt.show()

# Fig 8: common cohort - the same incidents scored at k = 1, 3, 5
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, target in zip(axes, TARGETS):
    for name in ['HistRate', 'LogReg', 'RandomForest', 'LightGBM', 'XGBoost', 'CatBoost']:
        d = T12[(T12.target == target) & (T12.model == name)]
        ax.plot(d.k, d.auc, color=COL[name], marker=MARK[name], lw=2, ms=5, label=name)
    n_inc = int(T12[(T12.target == target)].incidents.iloc[0])
    ax.set_title(f'{TNAME[target]} (same {n_inc:,} incidents)', loc='left', color=INK)
    ax.set_xticks([1, 3, 5]); ax.set_xlabel('Events observed so far (k)')
axes[0].set_ylabel('ROC-AUC'); axes[1].legend(frameon=False, fontsize=8, loc='lower right', ncol=2)
save_fig(fig, 'fig8_common_cohort.png'); plt.show()

## Step 15. Summary file

In [ ]:
def tbl(d, cols=None, nd=3):
    d = d if cols is None else d[cols]
    return d.round(nd).to_markdown(index=False) if hasattr(d, 'to_markdown') else d.round(nd).to_string(index=False)
try:
    import tabulate  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate'], check=True)
parts = [f"# OSMO ping-pong results\n\nRun settings: repeats={N_REPEATS}, folds={N_FOLDS}, max prefix={MAX_PREFIX}, alert budget={ALERT_BUDGET}",
         "## Environment\n```\n" + json.dumps(env, indent=1) + "\n```",
         "## T0a Segment-country ambiguity (with and without sender fix)\n" + tbl(T0a),
         "## T0b Cross-border ping-pong under four definitions\n" + tbl(T0b, nd=4),
         "## T0c Overlap between definitions (Jaccard, sender fix on)\n" + tbl(T0c, nd=2),
         "## T1 Prevalence and NAIVE resolution time (ever vs never)\n" + tbl(T1),
         f"NAIVE Mann-Whitney (cross-border vs none) p = {u.pvalue:.2e}; share of ping-pong crossing a border = {L.xpp.sum()/L.pp.sum():.3f}; any cross-border handover = {L.xh.mean():.3f}",
         "## T1c Time-varying Cox model (HR < 1 = slower resolution after exposure)\n" + tbl(T1c, nd=4),
         "## T1d Time-band proportionality check\n" + tbl(T1d, nd=4),
         "## T4b Rate by opening month\n" + tbl(month),
         "## T2 Country pairs (top 15)\n" + tbl(pairs.head(15)),
         "## T3 Rate by opening hour (UTC)\n" + tbl(hour),
         "## T4 Rate by first country\n" + tbl(cty),
         "## T5 Cross-validated results\n" + tbl(T5),
         "## T6 Per-prefix AUC\n" + tbl(T6.pivot_table(index=['target', 'k'], columns='model', values='auc').reset_index()),
         "## T7 Significance (Wilcoxon + Holm, Nadeau-Bengio corrected t-test)\n" + tbl(T7, nd=5),
         "## T11 Operating point at first event (threshold from training scores)\n" + tbl(T11),
         "## T12 Common cohort (same incidents at k = 1, 3, 5)\n" + tbl(T12.pivot_table(index=['target', 'model'], columns='k', values='auc').reset_index()),
         "## T13 Overfitting gap (train vs held-out AUC)\n" + tbl(T13),
         "## T14 Robustness (alternative labels, opening window, ablation)\n" + tbl(T14),
         "## T7b Configs chosen by inner tuning (count of folds)\n" + tbl(TUNE),
         "## T8 Temporal split\n" + tbl(T8),
         "## T9 SHAP (top 15)\n" + tbl(T9.head(15), nd=4),
         "## T10 Deployment cost\n" + tbl(T10)]
for c in ['first_team', 'product', 'first_country']:
    p = RES / f'T9_shap_by_{c}.csv'
    if p.exists():
        d = pd.read_csv(p)
        d = d if len(d) <= 16 else pd.concat([d.head(8), d.tail(8)])
        parts.append(f"## SHAP by {c} (top and bottom 8, n>=30)\n" + tbl(d, nd=4))
atomic_write(RES / 'RESULTS_SUMMARY.md', lambda p: pathlib.Path(p).write_text('\n\n'.join(parts)))
log('Results summary saved')
print('\n\n'.join(parts))

## Step 16. Zip everything and download it automatically
This cell runs by itself at the end. It packs the results summary, all result tables, all figures and the
progress log into one zip, keeps a copy in `MyDrive/OSMO_PingPong_replication/`, and downloads it to your computer.

**You can also run this cell on its own at any time** (click inside it, press **Shift + Enter**), for example
after stopping the run early. It then zips whatever is finished so far and adds `STATUS.md` listing what is
done and what is still missing.

In [ ]:
import os, pathlib, zipfile, time, re
try:
    from google.colab import drive, files
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE = pathlib.Path('/content/drive/MyDrive/OSMO_PingPong_replication'); IN_COLAB = True
except ImportError:
    BASE = pathlib.Path(os.environ.get('OSMO_BASE', './OSMO_PingPong_replication')); IN_COLAB = False

# progress report built from the saved pieces, so it works even if the run was stopped
expected_cv = None
try:
    expected_cv = len(TARGETS) * N_REPEATS * len(MODELS) * N_FOLDS
except NameError:
    pass
runs = BASE / 'runs'
cv_done = sorted(p.name for p in runs.glob('*__r*__f*.npz')) if runs.exists() else []
per = {}
for n in cv_done:
    t, m = n.split('__')[:2]; per[(t, m)] = per.get((t, m), 0) + 1
status = ['# Run status', f'Created: {time.strftime("%Y-%m-%d %H:%M:%S")}', '',
          f'Cross-validation pieces done: {len(cv_done)}' + (f' of {expected_cv}' if expected_cv else ''), '',
          '| target | model | folds done |', '|---|---|---|']
status += [f'| {t} | {m} | {c} |' for (t, m), c in sorted(per.items())]
status += ['', f'Temporal-split pieces done: {len(list(runs.glob("temporal__*.json")))}',
           f'Deployment-cost pieces done: {len(list(runs.glob("deploy__*.json")))}',
           f'Results summary present: {(BASE / "results" / "RESULTS_SUMMARY.md").exists()}',
           f'Robustness pieces done: {len(list(runs.glob("robust__*.npz")))}',
           f'Figures present: {len(list((BASE / "figures").glob("*.png")))} of 8']
(BASE / 'STATUS.md').write_text('\n'.join(status))
print('\n'.join(status))

stamp = time.strftime('%Y%m%d_%H%M')
zip_path = BASE / f'OSMO_results_{stamp}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ['results', 'figures']:
        for p in sorted((BASE / sub).glob('*')):
            if p.is_file() and not p.name.endswith('.part'):
                z.write(p, f'{sub}/{p.name}')
    for extra in ['progress_log.txt', 'STATUS.md']:
        if (BASE / extra).exists():
            z.write(BASE / extra, extra)
    if runs.exists():   # small per-piece result files, so partial runs are still usable
        for p in sorted(runs.glob('*.json')):
            z.write(p, f'runs/{p.name}')
print(f'\nZip saved in Drive: {zip_path}  ({zip_path.stat().st_size / 1024:.0f} KB)')
if IN_COLAB:
    files.download(str(zip_path))
    print('Download started. If your browser asks, click Allow. Send this zip back.')